In [1]:
import asyncio
import time
from collections import abc

from glow import timer, time_this, _wrap

_wrap._OP_CLBK = True
_wrap._OP_FORK_CORO = True
_wrap._OP_FORK_STOPITER = True
_wrap._OP_FUNC = True

DT = 0.1
FAIL = False

# thread cpu = busy, track thread_time
# sleep & syscalls = idle, track (Wall-thread_time)
# await = idle, track WAll
# intervals between __next__/send/throw = nothing

def _cpu100_busy1():
    # busy += dt
    end = time.perf_counter() + DT
    while time.perf_counter() < end:
        sum(range(1000))


def _sleep010_sys1():
    # idle += dt
    time.sleep(DT)


async def _asleep001_aw1():
    # idle += dt
    await asyncio.sleep(DT)


def work110_busy1_sys1():
    _cpu100_busy1()
    _sleep010_sys1()
    if FAIL:
        1 / 0


async def awork111_busy1_sys1_aw1():  # wall=3, frame=2, susp=1, cpu=1, io=2
    _cpu100_busy1()
    _sleep010_sys1()
    await _asleep001_aw1()
    if FAIL:
        1 / 0


class _Iter220_660:  # busy=2..6, sys=2..6, aw=0, 2/2/0 or 6/6/0
    def __init__(self) -> None:
        self.it = iter(range(0, 4, 2))

    def __iter__(self):
        return self

    def __next__(self) -> '_SubIter220':  # {busy=1, sys=1, aw=0} x2 + 2/2/0 x2
        i = self.it.__next__()
        work110_busy1_sys1()
        return _SubIter220(i)

    def __repr__(self):
        return f'<{self.__class__.__name__} at {id(self):x}>'


class _SubIter220:  # busy=2, sys=2, aw=0
    def __init__(self, i) -> None:
        self.it = iter(range(i, i + 2))

    def __iter__(self):
        return self

    def __next__(self):  # {busy=1, sys=1, aw=0} x2
        i = self.it.__next__()
        work110_busy1_sys1()
        return i

    def __repr__(self):
        return f'<{self.__class__.__name__} at {id(self):x}>'

In [2]:
tps = (
    abc.Iterator,
    abc.Generator,  # is iterator
    abc.Awaitable,
    abc.Coroutine,  # is awaitable
    abc.AsyncIterator,
    abc.AsyncGenerator,  # is async iterator
)


def _info[T](obj: T, tag: str = '') -> T:
    print(type(obj).__qualname__, obj, tag)

    bases = [
        tp.__name__
        for tp in tps
        if (
            issubclass(obj, tp)
            if isinstance(obj, type)
            else isinstance(obj, tp)
        )
    ]
    base = {
        *dir(object()),
        *dir(object),
        '__doc__',
        '__module__',
        '__class_getitem__',
        '__del__',
        '__name__',
        '__qualname__',
        '__dict__',
        '__weakref__',
    }
    if bases:
        print('- bases:', *bases)

    props = {k: callable(getattr(obj, k)) for k in sorted({*dir(obj)} - base)}
    if attrs := [k for k, ismethod in props.items() if not ismethod]:
        print('- attrs:', *attrs)
    if methods := [k for k, ismethod in props.items() if ismethod]:
        print('- methods:', *methods)
    return obj

In [3]:
async def adef() -> int:
    try:
        print('<sleep ...>')
        await asyncio.sleep(0.5)
        print('<sleep ok>')
        # async def a2():
        #     await asyncio.sleep(.1)
        #     1 / 0
        #     return 53
        # await asyncio.ensure_future(a2())
    except asyncio.CancelledError:
        print('<sleep err>')
        await asyncio.sleep(0.5)
        print('<sleep 2nd time>')
    except BaseException as e:
        print(f'<sleep base err {e!r}>')
        await asyncio.sleep(0.5)
        print('<sleep 2nd time>')
        raise
    return 42


aw = coro = _info(adef())  # -> coroutine: Coroutine & Awaitable


async def coro2():
    while True:
        try:
            f: asyncio.Future = _info(coro.send(None), '... <- await X')
        except StopIteration as e:
            return _info(e.value, '... <- return X')
        else:
            # _info(f.__await__(), 'X = _.__await__() <- await ...')

            ev = asyncio.Event()
            f.add_done_callback(lambda _, ev=ev: ev.set())

            try:
                await ev.wait()
            except BaseException as e:
                coro.throw(e)
                raise

            # try:
            #     while not f.done():
            #         await asyncio.sleep(0)
            # except BaseException as e:
            #     f.__await__().throw(e)
            #     raise


t = asyncio.create_task(coro2())
await asyncio.sleep(0.1)
t.cancel()

coroutine <coroutine object adef at 0x000001E0F1814AC0> 
- bases: Awaitable Coroutine
- attrs: cr_await cr_code cr_frame cr_origin cr_running cr_suspended
- methods: __await__ close send throw
<sleep ...>
Future <Future pending> ... <- await X
- bases: Awaitable
- attrs: _asyncio_future_blocking _callbacks _cancel_message _exception _log_traceback _loop _result _source_traceback _state
- methods: __await__ __iter__ _make_cancelled_error add_done_callback cancel cancelled done exception get_loop remove_done_callback result set_exception set_result


True

<sleep err>


In [4]:
@time_this
def fn_iter110_330_770():  # function -> iterator[int]
    work110_busy1_sys1()
    return _Iter220_660()


try:
    r_fn_iter = fn_iter110_330_770()
    print(':: called ::')
    print([x for xs in r_fn_iter for x in xs])
finally:
    fn_iter110_330_770.log_timing()

:: called ::
[0, 1, 2, 3]
85.00% -  266ms +  1.56s - __main__.fn_iter110_330_770 - 


In [37]:
async def fun():
    try:
        yield 'ok'
    # except GeneratorExit as ex:
    #     print(f'catched: {ex!r}')
    #     yield 'genex'
    except BaseException as ex:
        print(f'catched: {ex!r}')
        yield 'base'
    else:
        yield 'noexcept'
    yield 'done'

g = fun()
print(await anext(g))
# await g.aclose()
print(await g.athrow(GeneratorExit))  # ok, no rte
# print(await g.athrow(KeyError))  # ok
# print(await anext(g))  # ok
# await g.aclose()

ok
catched: GeneratorExit()
base


In [5]:
@time_this
def fn_gen110_330_770():  # function -> generator[int]
    work110_busy1_sys1()
    return (x for x in _Iter220_660())


try:
    r_fn_gen = fn_gen110_330_770()
    print(':: called ::')
    print([x for xs in r_fn_gen for x in xs])
finally:
    fn_gen110_330_770.log_timing()

:: called ::
[0, 1, 2, 3]
54.35% -  391ms +  1.24s - __main__.fn_gen110_330_770 - 


In [6]:
@time_this
def gen110_330_770():  # generator[int]
    work110_busy1_sys1()
    yield from _Iter220_660()


r_gen = gen110_330_770()
print(':: called ::')
try:
    print([x for xs in r_gen for x in xs])
finally:
    gen110_330_770.log_timing()

:: called ::
[0, 1, 2, 3]
31.34% -  328ms +  1.31s - __main__.gen110_330_770 - 


In [7]:
@time_this
async def coro111():  # coroutine[int]
    # await _asleep_wa1_fr0_sus1_cpu0_io1()
    # await asyncio.to_thread(_cpu_wa1_fr1_sus0_cpu1_io0)
    await awork111_busy1_sys1_aw1()
    return 42


r_coro = coro111()
print(':: called ::')
try:
    _info(await r_coro)
finally:
    coro111.log_timing()

:: called ::
<class 'coroutine'> -> StopIteration(<class 'int'>)
int 42 
- attrs: denominator imag numerator real
- methods: __abs__ __add__ __and__ __bool__ __ceil__ __divmod__ __float__ __floor__ __floordiv__ __getnewargs__ __index__ __int__ __invert__ __lshift__ __mod__ __mul__ __neg__ __or__ __pos__ __pow__ __radd__ __rand__ __rdivmod__ __rfloordiv__ __rlshift__ __rmod__ __rmul__ __ror__ __round__ __rpow__ __rrshift__ __rshift__ __rsub__ __rtruediv__ __rxor__ __sub__ __truediv__ __trunc__ __xor__ as_integer_ratio bit_count bit_length conjugate from_bytes is_integer to_bytes
 2.90% - 31.2ms +  287ms - __main__.coro111 - 


In [8]:
@time_this
async def coro_iter112_332_772():  # coroutine[iterator[int]]
    await awork111_busy1_sys1_aw1()
    await asyncio.ensure_future(asyncio.sleep(0.1))
    return _Iter220_660()  # ! not forked when cr.send threw StopIteration


r_coro_iter = coro_iter112_332_772()
print(':: called ::')
try:
    _info(aw := await r_coro_iter)
    print(':: awaited ::')
    print([x for xs in aw for x in xs])
finally:
    coro_iter112_332_772.log_timing()

:: called ::
<class 'coroutine'> -> StopIteration(<class '__main__._Iter220_660'>)
_Iterator <_Iter220_660 at 1e0f1768110> 
- bases: Iterator
- attrs: it
- methods: __iter__ __next__
:: awaited ::
[0, 1, 2, 3]
34.91% -  578ms +  1.09s - __main__.coro_iter112_332_772 - 


In [9]:
@time_this
async def coro_gen111_331_771():  # coroutine[generator[int]]
    await awork111_busy1_sys1_aw1()
    return (x for x in _Iter220_660())  # ! not forked when cr.send threw StopIteration


r_coro_gen = coro_gen111_331_771()
print(':: called ::')
try:
    _info(aw := await r_coro_gen)
    print(':: awaited ::')
    print([x for xs in aw for x in xs])
finally:
    coro_gen111_331_771.log_timing()

:: called ::
<class 'coroutine'> -> StopIteration(<class 'generator'>)
generator <generator object _as_gen at 0x000001E0F178EF80> 
- bases: Iterator Generator
- attrs: gi_code gi_frame gi_running gi_suspended gi_yieldfrom
- methods: __iter__ __next__ close send throw
:: awaited ::
[0, 1, 2, 3]
27.70% -  641ms +  878ms - __main__.coro_gen111_331_771 - 


In [10]:
@time_this
async def asyncgen011():  # async generator[int]
    await asyncio.sleep(DT)
    yield 0
    time.sleep(DT)
    yield 1
    # raise StopIteration(2)  # ! RuntimeError
    # raise StopAsyncIteration(2)  # ! RuntimeError
    # raise GeneratorExit  # ! ok


try:
    r_asyncgen = asyncgen011()
    print(':: called ::')
    print([x async for x in r_asyncgen])
finally:
    asyncgen011.log_timing()

:: called ::
<class 'async_generator_asend'> -__next__-> <class '_asyncio.Future'>
<class 'async_generator_asend'> -__next__-> StopIteration(<class 'int'>)
<class 'async_generator_asend'> -__next__-> StopIteration(<class 'int'>)
[0, 1]
 0.00% -     0s +  213ms - __main__.asyncgen011 - 
<class 'async_generator_asend'> -__next__-> StopIteration(<class 'int'>)
[0, 1]
 0.00% -     0s +  213ms - __main__.asyncgen011 - 


In [11]:
@time_this
async def asyncgen000():  # async generator[int]
    yield 0
    yield 1
    # return None  # ! SyntaxError
    # raise StopIteration(2)  # ! RuntimeError
    # raise StopAsyncIteration(2)  # ! RuntimeError
    # raise GeneratorExit  # ! ok


try:
    r_asyncgen = asyncgen000()
    print(':: called ::')
    print([x async for x in r_asyncgen])
finally:
    asyncgen000.log_timing()

:: called ::
<class 'async_generator_asend'> -__next__-> StopIteration(<class 'int'>)
<class 'async_generator_asend'> -__next__-> StopIteration(<class 'int'>)
[0, 1]
 0.00% -     0s + 19.6us - __main__.asyncgen000 - 


In [12]:
@time_this
async def asyncgen112_332_772():  # async generator[int]
    await awork111_busy1_sys1_aw1()
    await asyncio.gather(
        asyncio.ensure_future(asyncio.sleep(DT)),
        asyncio.sleep(0),
    )
    for x in _Iter220_660():
        yield x  # ! not forked as rvalue of ag.send


r_asyncgen = asyncgen112_332_772()
print(':: called ::')
try:
    print([x async for xs in r_asyncgen for x in xs])
finally:
    asyncgen112_332_772.log_timing()

:: called ::
<class 'async_generator_asend'> -__next__-> <class '_asyncio.Future'>
<class 'async_generator_asend'> -__next__-> <class 'asyncio.tasks._GatheringFuture'>
<class 'async_generator_asend'> -__next__-> StopIteration(<class '__main__._SubIter220'>)
<class 'async_generator_asend'> -__next__-> StopIteration(<class '__main__._SubIter220'>)
[0, 1, 2, 3]
22.28% -  672ms +  961ms - __main__.asyncgen112_332_772 - 


In [13]:
for obj in (
    r_fn_iter,
    r_fn_gen,
    r_gen,
    r_coro,
    r_coro_iter,
    r_coro_gen,
    r_asyncgen,
):
    print(obj, type(obj))
    # print(type(o).mro())
    # print(sorted({*dir(o)} - {*dir(object())} - {*dir(object)}
    #              - {'__del__', '__name__', '__qualname__'}))
    print('  mro:', [
        tp.__name__ for tp in (
            abc.Iterator,
            abc.Generator,  # is iterator
            abc.Awaitable,
            abc.Coroutine,  # is awaitable
            abc.AsyncIterator,
            abc.AsyncGenerator,  # is async iterator
        ) if isinstance(obj, tp)
    ])

<_Iter220_660 at 1e0e964d700> <class 'glow._wrap._Iterator'>
  mro: ['Iterator']
<generator object _as_gen at 0x000001E0F178D120> <class 'generator'>
  mro: ['Iterator', 'Generator']
<generator object _as_gen at 0x000001E0F178EB00> <class 'generator'>
  mro: ['Iterator', 'Generator']
<coroutine object _as_coroutine at 0x000001E0F178E7A0> <class 'coroutine'>
  mro: ['Awaitable', 'Coroutine']
<coroutine object _as_coroutine at 0x000001E0F178E680> <class 'coroutine'>
  mro: ['Awaitable', 'Coroutine']
<coroutine object _as_coroutine at 0x000001E0F178EC20> <class 'coroutine'>
  mro: ['Awaitable', 'Coroutine']
<async_generator object _as_asyncgen at 0x000001E0F175B670> <class 'async_generator'>
  mro: ['AsyncIterator', 'AsyncGenerator']


In [14]:
import asyncio
from glow import memoize

@memoize(3, batched=True)
async def func(xs):
    await asyncio.sleep(0.2)
    raise RuntimeError(xs)

async def sleepy(x, t):
    await asyncio.sleep(t)
    return await func([x])

async def catch():
    # await func([5, 5])
    await asyncio.gather(sleepy(5, 0.2), sleepy(5, 0.1))

await catch()

RuntimeError: [5]